In [1]:
import scanpy as sc

# Load the h5ad file
adata = sc.read_h5ad("SHARESEQ_brain_epitrace_obj_age_estimated_multiome.h5ad")

# Check the structure
print("Dataset shape:", adata.shape)
print("\nObservation columns (metadata):", adata.obs.columns.tolist())
print("\nReductions (e.g., UMAP):", adata.obsm.keys())

# Confirm critical fields exist
required = ["EpiTraceAgeiterative", "Cluster.Name", "cytotracerna"]
for field in required:
    exists = field in adata.obs.columns
    print(f"Field '{field}' exists: {exists}")

Dataset shape: (8981, 32581)

Observation columns (metadata): ['orig.ident', 'nCount_peaks', 'nFeature_peaks', 'cell', 'nCount_iterative', 'nFeature_iterative', 'EpiTraceAge_iterative', 'Accessibility_iterative', 'AccessibilitySmooth_iterative', 'EpiTraceAge_Clock_initial', 'Accessibility_initial', 'AccessibilitySmooth_initial', 'nCount_all', 'nFeature_all', 'all_snn_res.1', 'seurat_clusters', 'Cell.ID', 'Cluster.ID', 'Cluster.Name', 'celltype', 'cytotrace_rna', 'nCount_rna_spliced', 'nFeature_rna_spliced', 'nCount_rna_unspliced', 'nFeature_rna_unspliced', 'nCount_spliced', 'nFeature_spliced', 'nCount_unspliced', 'nFeature_unspliced']

Reductions (e.g., UMAP): KeysView(AxisArrays with keys: X_umap)
Field 'EpiTraceAgeiterative' exists: False
Field 'Cluster.Name' exists: True
Field 'cytotracerna' exists: False


In [5]:
# Rename the alias mapping 
# Use the actual column name found in dataset (cytotracerna)

print("Available columns:", adata.obs.columns.tolist())

Available columns: ['orig.ident', 'nCount_peaks', 'nFeature_peaks', 'cell', 'nCount_iterative', 'nFeature_iterative', 'EpiTraceAge_iterative', 'Accessibility_iterative', 'AccessibilitySmooth_iterative', 'EpiTraceAge_Clock_initial', 'Accessibility_initial', 'AccessibilitySmooth_initial', 'nCount_all', 'nFeature_all', 'all_snn_res.1', 'seurat_clusters', 'Cell.ID', 'Cluster.ID', 'Cluster.Name', 'celltype', 'cytotrace_rna', 'nCount_rna_spliced', 'nFeature_rna_spliced', 'nCount_rna_unspliced', 'nFeature_rna_unspliced', 'nCount_spliced', 'nFeature_spliced', 'nCount_unspliced', 'nFeature_unspliced']


In [ ]:
adata.obs['EpiTraceAgeiterative'] = adata.obs['EpiTraceAge_iterative']
adata.obs['cytotracerna'] = adata.obs['cytotrace_rna']
adata.obs['Cluster.Name'] = adata.obs['Cluster.Name'].astype('category')

In [20]:
import scvelo as scv
import cellrank as cr
from cellrank.kernels import VelocityKernel, CytoTRACEKernel
from cellrank.estimators import GPCCA
from anndata import AnnData
import numpy as np
import scipy.sparse as sp
from copy import copy
from typing import Any

class AgeKernel(cr.kernels.Kernel):
    def __init__(self, adata: AnnData, obs_key: str = "EpiTraceAgeiterative", **kwargs: Any):
        super().__init__(adata=adata, obs_key=obs_key, **kwargs)
    
    def _read_from_adata(self, obs_key: str, **kwargs: Any) -> None:
        super()._read_from_adata(**kwargs)
        self.pseudotime = 1 - self.adata.obs[obs_key].values
        
    def compute_transition_matrix(self, some_parameter: float = 0.5) -> "AgeKernel":
        if self._reuse_cache({"some_parameter": some_parameter}):
            return self
        
        transition_matrix = sp.diags((some_parameter,) * len(self.adata), dtype=np.float64)
        self.transition_matrix = transition_matrix
        return self

    def copy(self) -> "AgeKernel":
        return copy(self)

    def backward(self, **kwargs: Any) -> "AgeKernel":
        return self

In [21]:
# Align names
adata.obs['EpiTraceAgeiterative'] = adata.obs['EpiTraceAge_iterative']
adata.obs['cytotracerna'] = adata.obs['cytotrace_rna']
adata.obs['Cluster.Name'] = adata.obs['Cluster.Name'].astype('category')

# Preprocessing (as in original script)
adata.layers['spliced'] = adata.X
scv.pp.filter_and_normalize(adata)
scv.pp.moments(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata, random_state=0)
scv.tl.velocity(adata, mode='stochastic')
scv.tl.velocity_graph(adata)

Normalized count data: X, spliced, unspliced.


/scratch/4645957.1.l40s/ipykernel_3078438/1938664610.py:11: DeprecationWarning: Automatic neighbor calculation is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors first with Scanpy.
  scv.pp.moments(adata)
/usr4/ds722/tareqh/.local/lib/python3.12/site-packages/scvelo/preprocessing/moments.py:71: DeprecationWarning: `neighbors` is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors with Scanpy.
  neighbors(
/usr4/ds722/tareqh/.local/lib/python3.12/site-packages/scvelo/preprocessing/neighbors.py:233: DeprecationWarning: Automatic computation of PCA is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute PCA with Scanpy first.
  _set_pca(adata=adata, n_pcs=n_pcs, use_highly_variable=use_highly_variable)


computing neighbors
    finished (0:00:01) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:13) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
computing velocities


/usr4/ds722/tareqh/.local/lib/python3.12/site-packages/scvelo/tools/optimization.py:184: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gamma[i] = np.linalg.pinv(A.T.dot(A)).dot(A.T.dot(y[:, i]))


    finished (0:00:17) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)
computing velocity graph (using 1/32 cores)


  0%|          | 0/8981 [00:00<?, ?cells/s]

/share/pkg.8/academic-ml/fall-2025/install/fall-2025-pyt/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=3078438) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


    finished (0:00:15) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)


In [22]:
adata = adata[adata.obs['Cluster.Name'] != 8].copy()

In [31]:
import matplotlib.pyplot as plt
import scanpy as sc
import scvelo as scv
import numpy as np

adata = sc.read_h5ad("SHARESEQ_brain_epitrace_obj_age_estimated_multiome.h5ad")

adata.obs["EpiTraceAgeiterative"] = adata.obs["EpiTraceAge_iterative"]
adata.obs["cytotracerna"] = adata.obs["cytotrace_rna"]
adata.layers["spliced"] = adata.X.copy()

adata = adata[adata.obs["Cluster.Name"].isin([0,1,2,3,4,5,6,7])].copy()

mapping = {
    0: "Radial Glia",
    1: "Cyc. Prog",
    2: "nIPC",
    3: "ExN2",
    4: "ExN3",
    5: "ExN4",
    6: "ExN5",
    7: "mGPC/OPC",
}
adata.obs["Real.Name"] = adata.obs["Cluster.Name"].map(mapping).astype(str)

scv.pp.filter_and_normalize(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata, random_state=0)
scv.pp.moments(adata)
scv.tl.velocity(adata, mode="stochastic")
scv.tl.velocity_graph(adata)

color_list_a = ["#FFA500","#8B0000","#98F5FF","#8EE5EE","#7AC5CD","#53868B","#A2CD5A","#BCEE68"]

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="Real.Name",
    palette=color_list_a,
    title="Panel a: RNA velocity trajectory",
    density=2,
    dpi=400,
    figsize=(8,6),
    legend_loc="right",
    show=False
)
plt.savefig("panel_a_velocity_stream.png", dpi=400, bbox_inches="tight")
plt.close()

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

sc.pl.embedding(
    adata,
    basis="umap",
    color="cytotracerna",
    cmap="viridis",
    ax=axs[0],
    title="Panel b: CytoTRACE",
    show=False
)

sc.pl.embedding(
    adata,
    basis="umap",
    color="EpiTraceAgeiterative",
    cmap="autumn",
    ax=axs[1],
    title="Panel b: EpiTrace",
    show=False
)

plt.tight_layout()
plt.savefig("panel_b_cytotrace_epitrace.png", dpi=400, bbox_inches="tight")
plt.close()

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

sc.pl.embedding(
    adata,
    basis="umap",
    color="Real.Name",
    palette=color_list_a,
    ax=axs[0],
    title="Cell types",
    show=False
)

sc.pl.embedding(
    adata,
    basis="umap",
    color="EpiTraceAgeiterative",
    cmap="autumn",
    ax=axs[1],
    title="Mitotic age",
    show=False
)

plt.tight_layout()
plt.savefig("panel_a_combined_view.png", dpi=400, bbox_inches="tight")
plt.close()

print("Saved: panel_a_velocity_stream.png")
print("Saved: panel_b_cytotrace_epitrace.png")
print("Saved: panel_a_combined_view.png")

Normalized count data: X, spliced, unspliced.


/scratch/4645957.1.l40s/ipykernel_3078438/153604222.py:29: DeprecationWarning: Automatic neighbor calculation is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors first with Scanpy.
  scv.pp.moments(adata)
/usr4/ds722/tareqh/.local/lib/python3.12/site-packages/scvelo/preprocessing/moments.py:71: DeprecationWarning: `neighbors` is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors with Scanpy.
  neighbors(


computing neighbors
    finished (0:00:00) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:12) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
computing velocities


/usr4/ds722/tareqh/.local/lib/python3.12/site-packages/scvelo/tools/optimization.py:184: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gamma[i] = np.linalg.pinv(A.T.dot(A)).dot(A.T.dot(y[:, i]))


    finished (0:00:18) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)
computing velocity graph (using 1/32 cores)


  0%|          | 0/6720 [00:00<?, ?cells/s]

/share/pkg.8/academic-ml/fall-2025/install/fall-2025-pyt/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=3078438) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


    finished (0:00:35) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)
computing velocity embedding
    finished (0:00:00) --> added
    'velocity_umap', embedded velocity vectors (adata.obsm)
Saved: panel_a_velocity_stream.png
Saved: panel_b_cytotrace_epitrace.png
Saved: panel_a_combined_view.png


In [32]:
# Final Plotting Block for Figure 6a,b (Streamline version)

# Ensure the figure settings are set correctly for streamline visibility
scv.settings.set_figure_params("scvelo", dpi=400, fontsize=16)

# 1. Panel A: Combined Kernel (Figure 6a)
# Ensure combinedkernel is defined in your previous step
scv.pl.velocity_embedding_stream(
    adata, 
    basis="umap", 
    vkey="velocity", # This uses the graph you already computed
    color="Real.Name", 
    palette=["#FFA500","#8B0000","#98F5FF","#8EE5EE","#7AC5CD","#53868B","#A2CD5A","#BCEE68"],
    title="Panel A: Combined trajectory",
    density=1.5,
    figsize=(8,8),
    save="Figure6a_Streamlines.png",
    show=False
)

# 2. Panel B: Single Kernel Comparison
# Use the same function to get arrows for individual kernels
scv.pl.velocity_embedding_stream(
    adata, 
    basis="umap", 
    vkey="velocity", 
    color="Real.Name", 
    title="Panel B-i: RNA Velocity Only",
    density=1.5,
    figsize=(8,6),
    save="Figure6b_i_Velocity.png",
    show=False
)

print("Generated streamline plots: Figure6a_Streamlines.png and Figure6b_i_Velocity.png")

saving figure to file ./figures/scvelo_Figure6a_Streamlines.png
saving figure to file ./figures/scvelo_Figure6b_i_Velocity.png
Generated streamline plots: Figure6a_Streamlines.png and Figure6b_i_Velocity.png
